In [1]:
import pandas as pd

train = pd.read_csv("../data/processed/train_final.csv")
test = pd.read_csv("../data/processed/test_final.csv")

X_train = train.drop("habitable", axis=1)
y_train = train["habitable"]

X_test = test.drop("habitable", axis=1)
y_test = test["habitable"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1487, 6)
Test shape: (372, 6)


In [2]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

class_weight_dict

{np.int64(0): np.float64(0.5027045300878973), np.int64(1): np.float64(92.9375)}

In [4]:
X_train.isna().sum()

log_pl_orbsmax      0
log_st_lum          0
log_pl_rade         0
log_teq             0
log_st_teff         1
log_stellar_flux    0
dtype: int64

In [5]:
# Find rows with NaN
nan_rows = X_train[X_train.isna().any(axis=1)]

nan_rows

,log_pl_orbsmax,log_st_lum,log_pl_rade,log_teq,log_st_teff,log_stellar_flux
917,-0.392302,-1.022131,-1.251551,-0.374649,NaN,-0.688476


In [6]:
# Drop NaN rows
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

print("New train shape:", X_train.shape)

New train shape: (1486, 6)


In [9]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    class_weight=class_weight_dict,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully.")



Model trained successfully.


In [10]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

cm

array([[355,  15],
       [  0,   2]])

In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98       370
           1       0.12      1.00      0.21         2

    accuracy                           0.96       372
   macro avg       0.56      0.98      0.59       372
weighted avg       1.00      0.96      0.98       372

